In [21]:
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split

In [9]:
data = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

experiment_name = "house_prices"
server = "http://localhost:5000"

mlflow.set_tracking_uri(server)
mlflow.set_experiment(experiment_name)

X = data.drop(columns=["SalePrice", "Id"])
y = np.log1p(data["SalePrice"])

categorial_features = X.select_dtypes(include=["object", "category"]).columns.to_list()

numerical_features = X.select_dtypes(include= ["int64", "float64"]).columns.to_list()

for col in categorial_features:
    X[col] = X[col].fillna("Missing")
    
X_test = test.drop(columns=["Id"])

for col in categorial_features:
    X_test[col] = X_test[col].fillna("Missing")


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
skewed_cols = X_train[numerical_features].skew()
skewed_cols = skewed_cols[skewed_cols > 1].index.tolist()
 
for col in numerical_features:
    if np.issubdtype(X_train[col].dtype, np.number):
        if X_train[col].skew() > 1:
            X_train[col] = np.log1p(X_train[col])
            X_val[col]   = np.log1p(X_val[col])
            X_test[col]  = np.log1p(X_test[col])



In [16]:
cat_features = [X_train.columns.get_loc(col) for col in categorial_features]

In [17]:
model = CatBoostRegressor(
    iterations = 1000,
    learning_rate= 0.05,
    early_stopping_rounds=200,
    depth=6,
    verbose= 0,
    random_seed= 42
)

In [19]:
model.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features= cat_features)

In [24]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_val)
r2 = r2_score(y_val, y_pred)

# RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

# MAE
mae = mean_absolute_error(y_val, y_pred)

print(f"R²: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")


R²: 0.906
RMSE: 0.133
MAE: 0.088


In [ ]:
importances = model.get_feature_importance()
feature_names = X_train.columns


feat_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp_df.head(10))


         feature  importance
45     GrLivArea   15.676155
16   OverallQual   15.314765
52   KitchenQual    4.719438
37   TotalBsmtSF    4.630306
3        LotArea    4.067014
55    Fireplaces    3.751340
59  GarageFinish    3.123144
33    BsmtFinSF1    3.101026
18     YearBuilt    2.994229
42      1stFlrSF    2.713708


In [30]:

top_features = feat_imp_df.head(10)["feature"]


# Dataset réduit
X_train_top = X_train[top_features].copy()
X_val_top   = X_val[top_features].copy()
X_test_top  = X_test[top_features].copy()

# Identifier les features catégorielles dans ce top
cat_features_top = [i for i, col in enumerate(top_features) if col in categorial_features]

# Modèle CatBoost
model_top = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    early_stopping_rounds=200,
    verbose=0,
    random_seed=42
)

# Entraînement
model_top.fit(
    X_train_top, y_train,
    eval_set=(X_val_top, y_val),
    cat_features=cat_features_top
)

# Évaluation
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = model_top.predict(X_val_top)
r2 = r2_score(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f"R²: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")

R²: 0.875
RMSE: 0.153
MAE: 0.104
